# 30일 주가 예측 — GJR-GARCH(1,1,1) + Skew-t + FHS Monte Carlo Prototype

**목적**: `processor/feature4_chart_predict.py` 의 GARCH(1,1)-t + 단일 sigma 방식을 **GJR-GARCH(1,1,1) + Skew-t + Filtered Historical Simulation + 1년 윈도우 단축** 으로 업그레이드하기 전 노트북에서 prototype.

**3-Stage 비교**:
- **(A) baseline**: GARCH(1,1)-t, 5y window — 현재 운영 코드
- **(B) Stage 1**: GJR-GARCH(1,1,1)-Skew-t, 5y window — leverage + Skew-t 만
- **(C) Stage 1+2 (적용 대상)**: GJR-GARCH(1,1,1)-Skew-t, **1y window (252일)** + FHS Monte Carlo + Stage 3 API 페이로드 + Stage 4 visual cap

**왜 1년 윈도우?** — 이전 노트북 결과에서 5y window 사용 시 SPY 30일 p05~p95 range 가 -32% ~ +47% 로 비현실적으로 컸음 (코로나/2022 베어 포함). 1년 윈도우는 최근 변동성 환경만 반영 → tighter range.

**검증 기준**:
1. 시각: 평탄 line → fan + sample paths
2. 정량: AIC/BIC/MDA/Pinball Loss
3. 분포: skewness/kurtosis (fat-tail 형성 확인)
4. **Range realism**: 30일 p05~p95 가 ±10~15% 이내로 들어오는지
5. **API payload**: 본 코드에 그대로 붙여넣기 가능한 JSON 형태

**참고**: `processor/feature4_chart_predict.py:226-243` 의 `_garch_forecast` 함수가 본 적용 대상.

## 1. Setup & Imports

**의도**: 노트북에서 사용할 라이브러리 import + random seed 고정으로 재현성 확보.

**핵심 의존성**:
- `arch` — GJR-GARCH + Skew-t 적합 (`requirements.txt` 에 이미 존재)
- `xgboost` — mean path 학습 (이미 운영 코드 사용 중)
- `scipy.stats` — skewness/kurtosis 계산
- `matplotlib` — fan chart 시각화 (본 코드는 SVG path 직접 그리지만 노트북에선 matplotlib 으로 prototype)

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import json
import numpy as np
import pandas as pd
import yfinance as yf
import matplotlib.pyplot as plt
from scipy import stats

from arch import arch_model
from xgboost import XGBRegressor
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_pinball_loss, mean_squared_error, mean_absolute_error

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

## 2. Config

**의도**: 실험 파라미터 한곳에 모음. 본 코드 적용 시 그대로 옮길 상수.

**파라미터 의미**:
- `TICKER` — 'SPY'/'QQQ'/'AAPL' 등으로 바꿔서 sector ETF 별 검증 가능
- `HORIZON=30` — 30일 예측 (운영 동일)
- `N_SIMS=1000` — Monte Carlo path 개수. 본 코드 적용 시 동일
- `EMBARGO=30` — TimeSeriesSplit `gap` (look-ahead leakage 방지). plan Stage 5 핵심
- `HIST_PERIOD='5y'` — XGBoost 학습용 가격 시계열 (긴 데이터로 패턴 학습)
- **`GARCH_WINDOW=252`** — GJR-GARCH 적합 시 사용할 일별 데이터 길이 (1년 = ~252 거래일). **이것이 stage 1+2 의 핵심 튜닝**

In [ ]:
TICKER = 'SPY'
HORIZON = 30
N_SIMS = 1000
EMBARGO = 30
HIST_PERIOD = '5y'
GARCH_WINDOW = 252        # NEW: GJR-GARCH 적합용 윈도우 (1년)
VISUAL_CAP = 0.25         # NEW: chart 표시용 cap (median ± 25%)

## 3. Data Loading

**의도**: yfinance 로 5년치 OHLCV 받아서 종가 시계열 추출. 본 코드의 `run_chart_predict_single()` 와 동일.

**처리**:
- `auto_adjust=True` — 분할/배당 조정된 종가 사용
- 멀티인덱스 평탄화 (yfinance 가 가끔 columns 에 ticker 레벨 추가)
- 출력: `close: pd.Series`

In [ ]:
df = yf.download(TICKER, period=HIST_PERIOD, interval='1d', auto_adjust=True, progress=False)
if hasattr(df.columns, 'levels'):
    df.columns = df.columns.get_level_values(0)

close = df['Close']
print(f'{TICKER}: {len(close)} days, range {close.index[0].date()} ~ {close.index[-1].date()}')
print(f'last price: {float(close.iloc[-1]):.2f}')
close.tail()

## 4. Feature Engineering

**의도**: 본 코드 `build_features_v2()` 와 동일한 16개 피처. XGBoost 입력 `X` 와 1일 forward 로그수익률 타겟 `y` 생성.

**핵심 — leakage 방지**:
- 모든 rolling 계산에 `.shift(1)` 적용 — 오늘 값이 피처에 포함되지 않음
- target 은 `.shift(-1)` — 미래 1일 후 수익률
- `dropna()` 로 첫 200일 (SMA200 워밍업) 제거

**피처 카테고리**:
- 모멘텀: ret_lag_1/2/3/5/10/20
- 추세: sma_ratio_5/20/60/200, macd, roc_10
- 변동성: vol_5, vol_20
- 기술지표: rsi_14

In [ ]:
def build_features(close: pd.Series) -> pd.DataFrame:
    """피처 16개. 모두 .shift(1) 처리되어 오늘 값이 피처에 포함되지 않음."""
    feat = pd.DataFrame(index=close.index)
    log_ret = np.log(close / close.shift(1))

    for lag in [1, 2, 3, 5, 10, 20]:
        feat[f'ret_lag_{lag}'] = log_ret.shift(lag)

    for w in [5, 20, 60, 200]:
        feat[f'sma_ratio_{w}'] = (close / close.rolling(w).mean()).shift(1)

    feat['vol_5'] = log_ret.rolling(5).std().shift(1)
    feat['vol_20'] = log_ret.rolling(20).std().shift(1)

    delta = close.diff()
    gain = delta.clip(lower=0).rolling(14).mean()
    loss = -delta.clip(upper=0).rolling(14).mean()
    rs = gain / (loss + 1e-9)
    feat['rsi_14'] = (100 - 100 / (1 + rs)).shift(1)

    ema12 = close.ewm(span=12).mean()
    ema26 = close.ewm(span=26).mean()
    feat['macd'] = (ema12 - ema26).shift(1)

    feat['roc_10'] = (close.pct_change(10)).shift(1)
    return feat

features = build_features(close)
target = np.log(close / close.shift(1)).shift(-1)
target.name = 'target'
data = features.join(target).dropna()
X = data.drop(columns=['target'])
y = data['target']
print(f'X.shape: {X.shape}, y.shape: {y.shape}')
X.head(3)

## 5. XGBoost Training (mean path 학습)

**의도**: TimeSeriesSplit + embargo 로 leakage 없는 5-fold CV → 마지막 fold 모델을 prediction 에 사용.

**핵심 차이 vs 본 코드**:
- 본 코드는 **5-모델 앙상블** (XGB + CatBoost + RF + Ridge + SVR) — 평균 µ̂ 사용
- 노트북은 단순화 — XGBoost 단독으로 prototype (앙상블 효과는 본 적용 시 유지)
- `gap=30` (embargo) — 30일 forward target 사용하므로 train/val 의 시간 겹침 방지

**측정 메트릭**:
- `rmse` / `mae` — 학습 안정성 모니터
- **`mda`** (Mean Directional Accuracy) — 50% 가 random, 60%+ 면 우수. 비즈니스 가치 직결

In [ ]:
tscv = TimeSeriesSplit(n_splits=5, test_size=HORIZON, gap=EMBARGO)
fold_metrics = []
model = None

params = dict(
    objective='reg:squarederror', tree_method='hist', eval_metric='rmse',
    n_estimators=1000, early_stopping_rounds=50,
    learning_rate=0.02, max_depth=5,
    subsample=0.8, colsample_bytree=0.7, min_child_weight=5,
    reg_alpha=0.1, reg_lambda=1.0,
    random_state=RANDOM_STATE,
)

for fold, (tr, va) in enumerate(tscv.split(X)):
    m = XGBRegressor(**params)
    m.fit(X.iloc[tr], y.iloc[tr], eval_set=[(X.iloc[va], y.iloc[va])], verbose=False)
    pred = m.predict(X.iloc[va])
    rmse = np.sqrt(mean_squared_error(y.iloc[va], pred))
    mae = mean_absolute_error(y.iloc[va], pred)
    da = (np.sign(np.diff(y.iloc[va])) == np.sign(np.diff(pred))).mean()
    fold_metrics.append({'fold': fold, 'rmse': rmse, 'mae': mae, 'mda': da})
    print(f'fold {fold}: rmse={rmse:.5f}, mae={mae:.5f}, mda={da:.3f}')
    model = m

metrics_df = pd.DataFrame(fold_metrics)
print()
print(metrics_df.describe().loc[['mean', 'std']])

## 6. Recursive 30-step Mean Forecast

**의도**: XGBoost 가 평균 µ̂ 만 책임. 매 일차 최신 피처로 다음 일 수익률 예측 → history 업데이트 후 다음 step 반복.

**왜 recursive?** — 연구 문서 결론에 따라 **Monte Carlo + 잔차 부트스트랩과 가장 잘 결합**되는 멀티스텝 방식. Direct(30개 모델)보다 곡선이 부드럽고 학습 비용 낮음.

**입출력**:
- 입력: `model` (학습된 XGBoost), `close_history` (실측 종가 시계열), `horizon=30`
- 출력: `mu_path: np.ndarray (30,)` — 일별 mean log-return
- `np.exp(mu_path.sum())` 가 30일 누적 수익률 (1+r 형태)

In [ ]:
def recursive_mean_forecast(model, close_history: pd.Series, horizon: int, feature_cols):
    """30일 mean log-return path 생성."""
    hist = close_history.copy()
    mu_path = []
    for k in range(horizon):
        feat = build_features(hist)
        last_feat = feat.iloc[[-1]][feature_cols].values
        last_feat = np.nan_to_num(last_feat, nan=0.0, posinf=0.0, neginf=0.0)
        ret = float(model.predict(last_feat)[0])
        mu_path.append(ret)
        next_price = float(hist.iloc[-1]) * np.exp(ret)
        next_date = hist.index[-1] + pd.tseries.offsets.BDay(1)
        hist = pd.concat([hist, pd.Series([next_price], index=[next_date])])
    return np.array(mu_path)

mu_path = recursive_mean_forecast(model, close, HORIZON, list(X.columns))
print(f'mu_path shape: {mu_path.shape}, sum: {mu_path.sum():.4f} (= 누적 30일 log return)')
print(f'기대 30일 수익률: {(np.exp(mu_path.sum()) - 1) * 100:.2f}%')

## 7. Stage 1: GJR-GARCH(1,1,1) + Skew-t Fit (3 시나리오)

**의도**: arch 패키지로 변동성 모델 3가지 적합 후 AIC/BIC 비교.

**3 시나리오**:
1. **(A) baseline**: `GARCH(1,1) + Student-t`, **5y window** — 현재 운영 코드 line 231
2. **(B) Stage 1**: `GJR-GARCH(1,1,1) + Skew-t`, **5y window** — `o=1` + Skew-t 만 추가
3. **(C) Stage 1+2 적용 대상**: `GJR-GARCH(1,1,1) + Skew-t`, **1y window (252일)** — 본 코드 변경 후 모습

**왜 1년 윈도우?** — 이전 결과(5y)에서 SPY 30일 p05~p95 = -32%~+47% 로 비현실. 5y window 가 코로나/2022 베어 포함 → σ 부풀림. 1년이면 최근 vol regime 만 반영.

**arch 인자**:
- `o=1` — asymmetric leverage (음의 충격이 변동성에 더 강한 영향)
- `dist='skewt'` — Skew-t (fat-tail + 비대칭)
- `mean='Zero'` — 평균은 XGBoost 가 담당, GARCH 는 분산만

In [ ]:
log_ret_pct_full = (np.log(close / close.shift(1)).dropna()) * 100   # arch는 % 단위 권장
log_ret_pct_1y = log_ret_pct_full.iloc[-GARCH_WINDOW:]                # NEW: 1년 윈도우

# (A) baseline: GARCH(1,1) + Student-t, 5y
am_a = arch_model(log_ret_pct_full, vol='Garch', p=1, q=1, mean='Zero', dist='t')
res_a = am_a.fit(disp='off', show_warning=False)

# (B) Stage 1: GJR-GARCH(1,1,1) + Skew-t, 5y
am_b = arch_model(log_ret_pct_full, vol='Garch', p=1, o=1, q=1, mean='Zero', dist='skewt')
res_b = am_b.fit(disp='off', show_warning=False)

# (C) Stage 1+2 적용 대상: GJR-GARCH(1,1,1) + Skew-t, 1y (252일)
am_c = arch_model(log_ret_pct_1y, vol='Garch', p=1, o=1, q=1, mean='Zero', dist='skewt')
res_c = am_c.fit(disp='off', show_warning=False)

print('=== (A) GARCH(1,1)-t baseline (5y) ===')
print(res_a.params); print()
print('=== (B) GJR-GARCH-Skew-t (5y) ===')
print(res_b.params); print()
print('=== (C) GJR-GARCH-Skew-t (1y, 252일) — 적용 대상 ===')
print(res_c.params); print()
print(f'AIC: A={res_a.aic:.2f}, B={res_b.aic:.2f}, C={res_c.aic:.2f}  (낮을수록 좋음)')
print(f'BIC: A={res_a.bic:.2f}, B={res_b.bic:.2f}, C={res_c.bic:.2f}')
print('* C 의 샘플 수가 적어 (252) AIC 절대값이 낮은 건 자연스러움 — 분포 형태(skewness/kurtosis) 위주로 비교')

## 8. Stage 2: FHS Monte Carlo (3 시나리오 path 생성)

**의도**: 표준화 잔차 부트스트랩 + GJR variance 재귀 시뮬레이션 → N=1000 path × 30일.

**FHS 핵심 아이디어**:
1. 학습 기간 표준화 잔차 `z_t = ε_t / σ_t` 추출 → 분포 가정 없이 실제 종목의 fat-tail/비대칭을 직접 반영
2. 매 step `var_t = ω + α·ε² + γ·ε²·I[ε<0] + β·var_{t-1}` 갱신 (GJR)
3. 새 `ε_t = √var_t · z` (z는 historical std_resid 에서 부트스트랩)
4. mu_path[h] (XGBoost 평균) + GARCH-FHS noise 결합

**출력**: `paths_a/b/c: np.ndarray (1000, 30)` — 일별 log-return path × N_SIMS

**가격 복원**: `S0 * exp(cumsum(paths, axis=1))`

In [ ]:
def fhs_monte_carlo(res, mu_path: np.ndarray, n_sims: int, horizon: int):
    """표준화 잔차 부트스트랩 + GJR variance 재귀."""
    std_resid = (res.resid / res.conditional_volatility).dropna().values
    omega = res.params['omega']
    alpha = res.params.get('alpha[1]', 0.0)
    gamma = res.params.get('gamma[1]', 0.0)
    beta = res.params['beta[1]']
    last_var = float(res.conditional_volatility.iloc[-1] ** 2)
    last_eps = float(res.resid.iloc[-1])

    paths = np.zeros((n_sims, horizon))
    for s in range(n_sims):
        var_t, eps_t = last_var, last_eps
        for h in range(horizon):
            I = 1.0 if eps_t < 0 else 0.0
            var_t = omega + alpha * eps_t**2 + gamma * eps_t**2 * I + beta * var_t
            z = np.random.choice(std_resid)
            eps_t = np.sqrt(max(var_t, 1e-9)) * z
            paths[s, h] = mu_path[h] + eps_t / 100.0
    return paths

paths_a = fhs_monte_carlo(res_a, mu_path, N_SIMS, HORIZON)
paths_b = fhs_monte_carlo(res_b, mu_path, N_SIMS, HORIZON)
paths_c = fhs_monte_carlo(res_c, mu_path, N_SIMS, HORIZON)

S0 = float(close.iloc[-1])
price_a = S0 * np.exp(np.cumsum(paths_a, axis=1))
price_b = S0 * np.exp(np.cumsum(paths_b, axis=1))
price_c = S0 * np.exp(np.cumsum(paths_c, axis=1))

for label, prices in [('A baseline 5y', price_a), ('B GJR-Skew-t 5y', price_b), ('C GJR-Skew-t 1y', price_c)]:
    last = prices[:, -1]
    p05_pct = (np.percentile(last, 5) / S0 - 1) * 100
    p95_pct = (np.percentile(last, 95) / S0 - 1) * 100
    skw = stats.skew(last)
    krt = stats.kurtosis(last)
    print(f'{label:20s}  range 30d: [{p05_pct:+.2f}%, {p95_pct:+.2f}%]  skew={skw:+.3f}  kurt={krt:+.3f}')

## 9. Stage 3: API 응답 페이로드 시뮬레이션

**의도**: 본 코드 `api/routers/chart.py:/predict` 엔드포인트가 반환할 새 JSON 형태를 노트북에서 미리 만들어보고 검증.

**기존 응답** (변경 전):
```json
{ "predicted": [{"date": "...", "yhat": ..., "lower": ..., "upper": ...}] }
```

**새 응답** (Stage 3 적용 후):
- `forecast` — 일별 분위수 (p05/p10/p50/p90/p95) + mean/median
- `sample_paths` — 30개 path 샘플 (payload 크기 관리, 1000개 전체는 송출 X)
- `metrics` — 단일 점수 (expected_return / VaR 5% / prob_up_30d / prob_down_5pct)
- **하위 호환**: `predicted` 필드도 alias 로 유지 → 프론트 점진적 migrate

**JSON serialize 시 주의**: `np.float32`, `NaN`, `Inf` 는 직접 json.dumps 불가 → `_safe_float()` 헬퍼 (본 코드에 이미 존재)

In [ ]:
def _safe_float(v):
    """NaN/Inf -> None 으로 변환 (본 코드 _safe_float 와 동일)."""
    if v is None: return None
    f = float(v)
    if np.isnan(f) or np.isinf(f): return None
    return round(f, 4)

def build_response_payload(price_paths: np.ndarray, S0: float, ticker: str, last_date: pd.Timestamp,
                            horizon: int, n_sample_paths: int = 30) -> dict:
    """Stage 3 API 응답 형태로 직렬화.

    Returns: dict — JSON serializable. 본 코드 chart.py /predict 엔드포인트 응답 그대로.
    """
    n_sims = price_paths.shape[0]
    p05 = np.percentile(price_paths, 5, axis=0)
    p10 = np.percentile(price_paths, 10, axis=0)
    p50 = np.percentile(price_paths, 50, axis=0)
    p90 = np.percentile(price_paths, 90, axis=0)
    p95 = np.percentile(price_paths, 95, axis=0)
    mean_path = price_paths.mean(axis=0)
    future_dates = pd.bdate_range(last_date + pd.Timedelta(days=1), periods=horizon)

    forecast = []
    legacy_predicted = []  # 하위 호환 — 기존 yhat/lower/upper
    for i, d in enumerate(future_dates):
        date_str = d.strftime('%Y-%m-%d')
        forecast.append({
            'date': date_str,
            'mean': _safe_float(mean_path[i]),
            'median': _safe_float(p50[i]),
            'p05': _safe_float(p05[i]),
            'p10': _safe_float(p10[i]),
            'p90': _safe_float(p90[i]),
            'p95': _safe_float(p95[i]),
        })
        legacy_predicted.append({
            'date': date_str,
            'yhat': _safe_float(p50[i]),         # median == yhat alias
            'lower': _safe_float(p10[i]),         # 80% 신뢰구간 (기존과 비슷한 폭)
            'upper': _safe_float(p90[i]),
        })

    sample_idx = np.random.choice(n_sims, min(n_sample_paths, n_sims), replace=False)
    sample_paths = [[_safe_float(v) for v in price_paths[i].tolist()] for i in sample_idx]

    last_prices = price_paths[:, -1]
    ret_30d = last_prices / S0 - 1
    metrics = {
        'expected_return_30d': _safe_float(np.median(ret_30d)),
        'var_5pct_30d': _safe_float(np.percentile(ret_30d, 5)),
        'prob_up_30d': _safe_float((last_prices > S0).mean()),
        'prob_down_5pct_30d': _safe_float((ret_30d < -0.05).mean()),
    }

    return {
        'ticker': ticker,
        'last_price': _safe_float(S0),
        'last_date': last_date.strftime('%Y-%m-%d'),
        'horizon': horizon,
        'n_simulations': n_sims,
        'forecast': forecast,
        'sample_paths': sample_paths,
        'metrics': metrics,
        'predicted': legacy_predicted,            # 하위 호환
        'metadata': {
            'model': 'XGBoost+GJR-GARCH-SkewT-FHS',
            'garch_window_days': GARCH_WINDOW,
            'version': '2.0.0',
        },
    }

# 적용 대상 (C) 시나리오로 페이로드 생성
payload = build_response_payload(price_c, S0, TICKER, close.index[-1], HORIZON)

# 일부만 출력 (전체 직렬화 후 길이만 확인)
print(f'JSON 직렬화 OK — 길이 {len(json.dumps(payload))} bytes')
print()
print('forecast[0]:', json.dumps(payload['forecast'][0], indent=2, ensure_ascii=False))
print('forecast[-1]:', json.dumps(payload['forecast'][-1], indent=2, ensure_ascii=False))
print()
print('metrics:', json.dumps(payload['metrics'], indent=2, ensure_ascii=False))
print()
print(f'sample_paths: {len(payload["sample_paths"])} paths × {len(payload["sample_paths"][0])} days')

## 10. Stage 4: Fan Chart with Visual Cap (3 시나리오 비교)

**의도**: 본 코드 `static/js/chart.js` 의 SVG path 렌더링 로직을 matplotlib 으로 prototype.

**4-layer fan chart**:
1. **외측 밴드** (p05~p95): `alpha=0.10`, 옅은 파랑 — 95% 신뢰
2. **내측 밴드** (p10~p90): `alpha=0.22`, 짙은 파랑 — 80% 신뢰
3. **샘플 30개 path**: `alpha=0.12`, 가는 회색 — 'live' 한 느낌
4. **median 실선**: `lw=2`, 진한 파랑 — 중심 추정

**Stage 4 핵심 — Visual Cap**:
- 모든 path / 분위수를 `[median * (1 - VISUAL_CAP), median * (1 + VISUAL_CAP)]` 로 clip
- `VISUAL_CAP = 0.25` → 차트 표시는 ±25% 안으로 제한
- 통계적으로는 outlier path 가 존재하지만 **시각화에서만** cap (저장된 raw paths 는 그대로)

**각 시나리오 한 panel 씩**: A/B/C 의 시각 차이 비교

In [ ]:
def apply_visual_cap(prices: np.ndarray, S0: float, cap: float = 0.25) -> np.ndarray:
    """median 기준 ±cap% 로 clip — 차트 표시용. raw data 는 변경 X."""
    p50 = np.percentile(prices, 50, axis=0)
    lo = p50 * (1 - cap)
    hi = p50 * (1 + cap)
    return np.clip(prices, lo, hi)

future_dates = pd.bdate_range(close.index[-1] + pd.Timedelta(days=1), periods=HORIZON)
history_dates = close.index[-90:]
history_prices = close.iloc[-90:].values

fig, axes = plt.subplots(1, 3, figsize=(20, 6), sharey=True)
scenarios = [
    (axes[0], price_a, '(A) GARCH(1,1)-t baseline 5y'),
    (axes[1], price_b, '(B) GJR-GARCH-Skew-t 5y'),
    (axes[2], price_c, '(C) GJR-GARCH-Skew-t 1y + visual cap'),
]

for ax, prices_raw, title in scenarios:
    # Stage 4 visual cap 만 (C) 에 적용 — A/B 는 raw 표시 (비교용)
    prices = apply_visual_cap(prices_raw, S0, VISUAL_CAP) if 'visual cap' in title else prices_raw

    p05 = np.percentile(prices, 5, axis=0)
    p10 = np.percentile(prices, 10, axis=0)
    p50 = np.percentile(prices, 50, axis=0)
    p90 = np.percentile(prices, 90, axis=0)
    p95 = np.percentile(prices, 95, axis=0)

    ax.fill_between(future_dates, p05, p95, alpha=0.10, color='#3B82F6', label='p05~p95 (95%)')
    ax.fill_between(future_dates, p10, p90, alpha=0.22, color='#3B82F6', label='p10~p90 (80%)')
    sample_idx = np.random.choice(N_SIMS, 30, replace=False)
    for i in sample_idx:
        ax.plot(future_dates, prices[i], color='gray', alpha=0.12, lw=0.5)
    ax.plot(future_dates, p50, color='#3B82F6', lw=2, label='median')
    ax.plot(history_dates, history_prices, color='#1f2937', lw=1.2, label='actual')
    ax.axvline(close.index[-1], color='red', ls='--', alpha=0.5)
    ax.set_title(title)
    ax.legend(loc='upper left', fontsize=9)
    ax.grid(alpha=0.3)

plt.suptitle(f'{TICKER} — 30 Day Forecast (S0 = {S0:.2f})', y=1.02, fontsize=14)
plt.tight_layout()
plt.show()

## 11. 검증 — Walk-forward 정량 평가

**의도**: 5-fold walk-forward + Pinball Loss + MDA 측정해 plan stage 5 검증 시뮬레이션.

**Pinball Loss** (α=0.05, 0.95):
- α=0.05 quantile 의 cost — 실제 값이 분위수보다 낮으면 더 큰 cost
- 분위수 정확도 직접 측정 (RMSE 보다 fan chart calibration 에 적합)

**평가 방법**:
- 마지막 30일 holdout (이미 학습에 빠짐)
- 1y window FHS 로 5000 path 시뮬레이션 → 분위수 추출 → 실제 holdout 가격과 비교

In [ ]:
# 마지막 30일 holdout (이미 model 학습 시 TimeSeriesSplit 의 test fold 였음)
holdout = close.iloc[-HORIZON:]
actual_log_ret = np.log(holdout / float(close.iloc[-HORIZON - 1])).values  # cumulative log-return at each day

# 시나리오 (C) paths 의 cumulative log-return
cum_log_ret = np.cumsum(paths_c, axis=1)  # (1000, 30)

p05_q = np.percentile(cum_log_ret, 5, axis=0)
p50_q = np.percentile(cum_log_ret, 50, axis=0)
p95_q = np.percentile(cum_log_ret, 95, axis=0)

# RMSE (median 기준)
rmse_med = np.sqrt(((actual_log_ret - p50_q) ** 2).mean())
# MDA (방향성)
mda = (np.sign(np.diff(actual_log_ret)) == np.sign(np.diff(p50_q))).mean()
# Pinball Loss
pinball_05 = mean_pinball_loss(actual_log_ret, p05_q, alpha=0.05)
pinball_95 = mean_pinball_loss(actual_log_ret, p95_q, alpha=0.95)

# Coverage — 실제값이 80% 신뢰구간 (p10~p90) 안에 들어온 비율
p10_q = np.percentile(cum_log_ret, 10, axis=0)
p90_q = np.percentile(cum_log_ret, 90, axis=0)
coverage_80 = ((actual_log_ret >= p10_q) & (actual_log_ret <= p90_q)).mean()
coverage_90 = ((actual_log_ret >= p05_q) & (actual_log_ret <= p95_q)).mean()

print('=== (C) GJR-Skew-t 1y window — holdout 30일 평가 ===')
print(f'RMSE (median):       {rmse_med:.5f}')
print(f'MDA (방향성):        {mda:.3f}  (0.50 = random)')
print(f'Pinball Loss α=0.05: {pinball_05:.5f}')
print(f'Pinball Loss α=0.95: {pinball_95:.5f}')
print(f'80% coverage:        {coverage_80:.3f}  (이상적: 0.80)')
print(f'90% coverage:        {coverage_90:.3f}  (이상적: 0.90)')
print()
print('주의: holdout 1개 sample 만의 평가 — 실제 stage 5 walk-forward 5-fold 에서는 5개 샘플 평균')

## 12. 본 코드 적용 가이드

**노트북 검증 후 본 코드 적용 단계** (plan Stage 1-4):

### Stage 1+2 — `processor/feature4_chart_predict.py:_garch_forecast()` 재작성

변경 라인 (line 226-243):
```python
# AS-IS
log_ret = (np.log(close_history / close_history.shift(1)).dropna()) * 100
am = arch_model(log_ret, vol='Garch', p=1, q=1, mean='Zero', dist='t')

# TO-BE
log_ret = (np.log(close_history / close_history.shift(1)).dropna()) * 100
log_ret_window = log_ret.iloc[-252:]            # 1년 윈도우
am = arch_model(log_ret_window, vol='Garch', p=1, o=1, q=1, mean='Zero', dist='skewt')
```

**Hard clip 제거** (line 213-224, 262-274):
- `daily_clip`, `price_band`, `ci_band` 삭제
- `scaled_ret * 3.0` 삭제 (땜방 증폭 제거)
- 3일 MA smoothing 삭제
- 대신 path 단위 sanity bound 만: `np.clip(daily_log_ret, -0.10, 0.10)`

**FHS Monte Carlo 추가** — 위 노트북 cell 8 (`fhs_monte_carlo`) 그대로.

### Stage 3 — `api/routers/chart.py:/predict` 응답 확장

위 노트북 cell 18 (`build_response_payload`) 그대로 옮김. `legacy_predicted` 필드로 하위 호환.

### Stage 4 — `static/js/chart.js` fan chart SVG

현재 단일 밴드:
```javascript
predSvg += `<path d="M${upper} L${lower} Z" fill="rgba(59,130,246,0.12)"/>`;
```

→ 4-layer 추가:
```javascript
// Layer 1: p05~p95 (옅은 파랑)
predSvg += `<path d="M${p95Path} L${p05PathRev} Z" fill="rgba(59,130,246,0.06)"/>`;
// Layer 2: p10~p90 (짙은 파랑)
predSvg += `<path d="M${p90Path} L${p10PathRev} Z" fill="rgba(59,130,246,0.14)"/>`;
// Layer 3: sample 30 paths (가는 회색)
samplePaths.forEach(path => predSvg += `<path d="M${path}" fill="none" stroke="rgba(255,255,255,0.06)" stroke-width="0.5"/>`);
// Layer 4: median (진한 파랑 실선)
predSvg += `<path d="M${p50Path}" fill="none" stroke="#3B82F6" stroke-width="2"/>`;
```

**Visual cap 적용**: chart.js 에서 SVG 좌표 계산 전에 median±25% clip.

### 체크리스트

- [ ] 1년 윈도우 (`GARCH_WINDOW = 252`) range 가 ±10~15% 이내인지 확인
- [ ] AIC/BIC 가 (A) 대비 명확히 낮은지 (Skew-t λ < 0 + γ > α 가 일관되게 나타나면 OK)
- [ ] sample_paths 30개로 줄여 payload 크기 ≤ 30KB 유지
- [ ] `legacy_predicted` 필드로 `chart.js` 의 `_predictData.predicted` 가 그대로 작동
- [ ] Walk-forward 5-fold 에서 80% coverage ≈ 0.80 (calibration 확인)

**참고 파일**:
- 본 코드: [processor/feature4_chart_predict.py](../processor/feature4_chart_predict.py)
- API: [api/routers/chart.py](../api/routers/chart.py)
- 프론트: [static/js/chart.js](../static/js/chart.js)
- Plan: `/root/.claude/plans/pure-chasing-perlis.md`

## 13. 백엔드 production 로직으로 QQQ 10년 Walk-Forward Backtest

**의도**: `processor/feature4_chart_predict.py` 의 **현재 운영 30일 예측 함수** (`_train_models` + `_garch_forecast`) 를 그대로 import 해서 QQQ 10년치에 walk-forward 적용. 사용자가 매 시점 production 으로 받는 30일 예측이 실제 가격과 얼마나 일치했는지 시각 비교.

**왜 직접 import?** — 노트북에서 production 동일 동작 보장. 본 코드 변경 시 자동 동기화 (단, `sys.path.insert` 로 프로젝트 루트 추가).

**Walk-forward 절차**:
1. QQQ 15년치 종가 다운로드 (학습용 5년 + 평가용 10년)
2. 매 분기 1번 cutoff 시점 t 선택 (10년 = 약 40 시점)
3. 각 cutoff 에서 `close[:t]` 만 사용해 5-모델 앙상블 학습 + 30일 예측
4. 예측 결과 (`yhat`/`lower`/`upper`) 와 실제 30일 종가 (`close[t:t+30]`) 모두 저장
5. 다음 셀에서 이중축 plot + stats

**비용 추정**: 분기 1번 × 40 시점 × 5모델 학습 ≈ 5–15분. 매월로 늘리면 120 시점 × 5모델 ≈ 15–30분.

**주의**:
- 매 cutoff 마다 production `_train_models` 가 호출됨 — 실제 사이트의 매 사이클 동작 그대로
- `_garch_forecast` 의 hard-clipping (`daily_clip`, `price_band`, 3일 MA smoothing) 도 그대로 — 즉 **현재 운영 모델의 한계**가 그대로 노출됨

In [ ]:
import sys
sys.path.insert(0, '/root/Passive-Financial-Data-Analysis')
from processor.feature4_chart_predict import build_features_v2, _train_models, _garch_forecast

TICKER_BT = 'QQQ'
BT_HORIZON = 30
BT_PERIOD = '15y'
BT_EVAL_YEARS = 10
BT_CUTOFF_FREQ = 'ME'         # 분기 말 (10년 = ~40 시점). 'ME' = 매월(120점), '6ME' = 6개월(20점)

df_bt = yf.download(TICKER_BT, period=BT_PERIOD, interval='1d', auto_adjust=True, progress=False)
if hasattr(df_bt.columns, 'levels'):
    df_bt.columns = df_bt.columns.get_level_values(0)
close_full = df_bt['Close']
print(f'{TICKER_BT}: {len(close_full)} days, {close_full.index[0].date()} ~ {close_full.index[-1].date()}')

walkfw_start = close_full.index[-1] - pd.Timedelta(days=365 * BT_EVAL_YEARS)
walkfw_end = close_full.index[-1] - pd.Timedelta(days=BT_HORIZON + 5)
cutoff_dates = pd.date_range(walkfw_start, walkfw_end, freq=BT_CUTOFF_FREQ)
print(f'walk-forward cutoff: {len(cutoff_dates)} 시점 ({BT_CUTOFF_FREQ})')

bt_records = []
for i, cutoff in enumerate(cutoff_dates):
    cutoff_actual = close_full.loc[close_full.index <= cutoff]
    if len(cutoff_actual) < 300:
        continue
    try:
        feats = build_features_v2(cutoff_actual)
        tgt = np.log(cutoff_actual / cutoff_actual.shift(1)).shift(-1)
        tgt.name = 'target'
        data_bt = feats.join(tgt).dropna()
        if len(data_bt) < 200:
            continue
        feature_cols = list(feats.columns)
        X_bt = data_bt[feature_cols]
        y_bt = data_bt['target']
        models_bt = _train_models(X_bt, y_bt)
        preds = _garch_forecast(models_bt, cutoff_actual, BT_HORIZON, feature_cols, ticker=TICKER_BT)
        actual_after = close_full.loc[close_full.index > cutoff].head(BT_HORIZON)
        bt_records.append({
            'cutoff': cutoff,
            'cutoff_price': float(cutoff_actual.iloc[-1]),
            'pred_dates': pd.to_datetime([p['date'] for p in preds]),
            'pred_yhat': np.array([p['yhat'] for p in preds]),
            'pred_lower': np.array([p['lower'] for p in preds]),
            'pred_upper': np.array([p['upper'] for p in preds]),
            'actual_dates': actual_after.index,
            'actual_prices': actual_after.values,
        })
        actual_str = f'{actual_after.iloc[-1]:.2f}' if len(actual_after) >= BT_HORIZON else 'N/A'
        print(f'  [{i+1:2d}/{len(cutoff_dates)}] cutoff={cutoff.date()}  cutoff_price={cutoff_actual.iloc[-1]:.2f}  pred_30d={preds[-1]["yhat"]:.2f}  actual_30d={actual_str}')
    except Exception as e:
        print(f'  [{i+1}] cutoff={cutoff.date()} 실패: {e}')
        continue

print(f'\n총 {len(bt_records)} 시점 backtest 완료')

## 14. 이중축 비교 차트 + Backtest Stats

**의도**: 사용자 요청 — QQQ 10년 실제 차트 vs production 예측 차트 동시 비교.

**이중축 구성**:
- **왼쪽 Y축 (Price)**: 실제 종가 (검은 실선) + 매 cutoff 의 30일 예측 fragment (파란 선, alpha=0.5) + 각 fragment 의 lower/upper 신뢰구간 (옅은 파랑 영역)
- **오른쪽 Y축 (Error %)**: 매 cutoff 의 30일 후 예측-실제 오차 (%) — bar 형태. 빨강=과대예측, 초록=과소예측

**Stats 출력**:
- 30일 후 가격 오차: 평균/표준편차/MAE/RMSE
- 방향성 정확도 (cutoff 대비 30일 후 상승/하락 일치 비율)
- VaR coverage — 실제값이 [lower, upper] 안에 들어온 비율 (이상적: 80% 신뢰구간이면 0.80)

**해석 가이드**:
- 예측 fragment 가 실제 line 과 일관된 방향이면 추세 추종 OK
- error bar 가 bull market(2020-2021) 에서 음수(과소), bear market(2022) 에서 양수(과대) 패턴이면 — XGBoost 가 평균 회귀 경향이 있다는 증거
- coverage 가 0.80 보다 크게 낮으면 신뢰구간이 너무 좁음 (현재 hard-clip 의 단점), 너무 높으면 (>0.95) 너무 넓음

In [ ]:
if not bt_records:
    print('backtest 결과 없음 — 이전 셀 먼저 실행')
else:
    fig, ax1 = plt.subplots(figsize=(18, 8))

    plot_window = close_full.loc[close_full.index >= walkfw_start - pd.Timedelta(days=180)]
    ax1.plot(plot_window.index, plot_window.values, color='#1f2937', lw=1.2, label=f'{TICKER_BT} actual', zorder=3)

    for r in bt_records:
        ax1.fill_between(r['pred_dates'], r['pred_lower'], r['pred_upper'], color='#3B82F6', alpha=0.08, zorder=1)
        ax1.plot(r['pred_dates'], r['pred_yhat'], color='#3B82F6', lw=0.9, alpha=0.55, zorder=2)

    ax1.set_xlabel('Date')
    ax1.set_ylabel(f'{TICKER_BT} Price ($)', color='#1f2937', fontsize=11)
    ax1.tick_params(axis='y', labelcolor='#1f2937')
    ax1.grid(alpha=0.3)

    ax2 = ax1.twinx()
    err_dates, err_pcts = [], []
    for r in bt_records:
        if len(r['actual_prices']) >= BT_HORIZON:
            pred_30d = r['pred_yhat'][-1]
            actual_30d = r['actual_prices'][-1]
            err_pct = (pred_30d - actual_30d) / actual_30d * 100
            err_dates.append(r['actual_dates'][-1])
            err_pcts.append(err_pct)

    err_pcts_arr = np.array(err_pcts)
    bar_colors = ['#EF4444' if e > 0 else '#10B981' for e in err_pcts_arr]
    ax2.bar(err_dates, err_pcts_arr, width=20, color=bar_colors, alpha=0.45, label='30d Pred Error %')
    ax2.axhline(0, color='gray', lw=0.5, ls='--')
    ax2.set_ylabel('Prediction Error % (red=over, green=under)', color='#6b7280', fontsize=11)
    ax2.tick_params(axis='y', labelcolor='#6b7280')

    h1, l1 = ax1.get_legend_handles_labels()
    h2, l2 = ax2.get_legend_handles_labels()
    ax1.legend(h1 + h2 + [plt.Line2D([], [], color='#3B82F6', lw=0.9, alpha=0.55)],
                l1 + l2 + ['30d prediction (per cutoff)'],
                loc='upper left', fontsize=10)

    plt.title(f'{TICKER_BT} {BT_EVAL_YEARS}년 Walk-Forward Backtest — Production 30일 예측 vs 실제 (cutoff: {BT_CUTOFF_FREQ})')
    plt.tight_layout()
    plt.show()

    print('\n=== Backtest Stats ===')
    print(f'평가 시점: {len(err_pcts_arr)} 개 cutoff (30일 forward 모두 보유)')
    print(f'\n[30일 후 가격 오차 %]')
    print(f'  mean : {err_pcts_arr.mean():+.2f}%  (양수 = 평균적으로 과대예측, 음수 = 과소예측)')
    print(f'  std  : {err_pcts_arr.std():.2f}%')
    print(f'  MAE  : {np.abs(err_pcts_arr).mean():.2f}%')
    print(f'  RMSE : {np.sqrt((err_pcts_arr**2).mean()):.2f}%')
    print(f'  median: {np.median(err_pcts_arr):+.2f}%')

    pred_dirs, actual_dirs = [], []
    for r in bt_records:
        if len(r['actual_prices']) >= BT_HORIZON:
            pred_dir = np.sign(r['pred_yhat'][-1] - r['cutoff_price'])
            actual_dir = np.sign(r['actual_prices'][-1] - r['cutoff_price'])
            pred_dirs.append(pred_dir); actual_dirs.append(actual_dir)
    dir_accuracy = (np.array(pred_dirs) == np.array(actual_dirs)).mean()
    print(f'\n[방향성 정확도 (30일 후 상승/하락 일치)]')
    print(f'  hit rate: {dir_accuracy:.3f}  ({sum(np.array(pred_dirs) == np.array(actual_dirs))}/{len(pred_dirs)} 시점)')

    covered = []
    for r in bt_records:
        if len(r['actual_prices']) >= BT_HORIZON:
            actual_30d = r['actual_prices'][-1]
            if r['pred_lower'][-1] <= actual_30d <= r['pred_upper'][-1]:
                covered.append(1)
            else:
                covered.append(0)
    coverage = np.mean(covered)
    print(f'\n[Coverage]')
    print(f'  coverage: {coverage:.3f}  (현재 _garch_forecast 신뢰구간 calibration 진단)')

## 15. 하락 예측 분석 (Downside Detection)

**의도**: 39 cutoff 중 실제 30일 후 **하락한 사례** 만 따로 봐서 production 모델이 하락을 얼마나 잘 잡아냈는지 진단. 사용자에게 손실 방지가 가장 중요하므로 hit rate 평균(0.692) 보다 더 중요한 metric.

**Confusion matrix** (방향성):
- **TP** (True Positive): 실제 상승 + 모델 상승 예측 → 정상
- **TN** (True Negative): 실제 하락 + **모델 하락 예측** → ⭐ 사용자가 알고 싶은 case
- **FP** (False Positive): 실제 하락 + 모델 상승 예측 → ⚠️ 손실 위험 (모델이 안심시키는데 떨어짐)
- **FN** (False Negative): 실제 상승 + 모델 하락 예측 → 기회 손실

**핵심 metric**:
- **Specificity (TNR)** = TN / (TN + FP) — 실제 하락 중 모델이 잡아낸 비율
- **Precision (down)** = TN / (TN + FN) — 모델이 하락 예측한 것 중 실제 하락 비율
- **하락 예측 시 평균 손실 절감** — 모델이 하락 잡았을 때 실제 얼마나 빠졌는지

In [ ]:
if not bt_records:
    print('backtest 결과 없음 — cell 25 먼저 실행')
else:
    rows = []
    for r in bt_records:
        if len(r['actual_prices']) < BT_HORIZON:
            continue
        cp = r['cutoff_price']
        ap = r['actual_prices'][-1]
        pp = r['pred_yhat'][-1]
        actual_ret = (ap - cp) / cp
        pred_ret = (pp - cp) / cp
        rows.append({
            'cutoff': r['cutoff'].date(),
            'cutoff_price': cp,
            'actual_30d': ap,
            'pred_30d': pp,
            'actual_ret_pct': actual_ret * 100,
            'pred_ret_pct': pred_ret * 100,
            'actual_dir': 'down' if actual_ret < 0 else 'up',
            'pred_dir': 'down' if pred_ret < 0 else 'up',
        })
    bt_df = pd.DataFrame(rows)

    # confusion matrix
    TP = ((bt_df['actual_dir'] == 'up') & (bt_df['pred_dir'] == 'up')).sum()
    TN = ((bt_df['actual_dir'] == 'down') & (bt_df['pred_dir'] == 'down')).sum()
    FP = ((bt_df['actual_dir'] == 'down') & (bt_df['pred_dir'] == 'up')).sum()
    FN = ((bt_df['actual_dir'] == 'up') & (bt_df['pred_dir'] == 'down')).sum()

    n_total = len(bt_df)
    n_up_actual = (bt_df['actual_dir'] == 'up').sum()
    n_down_actual = (bt_df['actual_dir'] == 'down').sum()
    n_up_pred = (bt_df['pred_dir'] == 'up').sum()
    n_down_pred = (bt_df['pred_dir'] == 'down').sum()

    print(f'=== Class distribution (실제 30일 후) ===')
    print(f'  up:    {n_up_actual:2d} / {n_total} ({n_up_actual/n_total:.1%})')
    print(f'  down:  {n_down_actual:2d} / {n_total} ({n_down_actual/n_total:.1%})')
    print()
    print(f'=== Class distribution (모델 예측) ===')
    print(f'  up:    {n_up_pred:2d} / {n_total} ({n_up_pred/n_total:.1%})')
    print(f'  down:  {n_down_pred:2d} / {n_total} ({n_down_pred/n_total:.1%})')
    print()
    print(f'=== Confusion Matrix ===')
    print(f'                  pred up    pred down')
    print(f'  actual up      {TP:3d} (TP)    {FN:3d} (FN)')
    print(f'  actual down    {FP:3d} (FP)    {TN:3d} (TN)  <-- 하락 예측 적중')
    print()

    print('=== 하락 예측 metrics ===')
    if (TN + FP) > 0:
        specificity = TN / (TN + FP)
        print(f'  Specificity (TNR)  = TN/(TN+FP) = {TN}/{TN+FP} = {specificity:.3f}')
        print(f'    → 실제 하락한 {TN+FP} 시점 중 모델이 {TN} 개를 잡아냄')
    if (TN + FN) > 0:
        precision_down = TN / (TN + FN)
        print(f'  Precision (down)   = TN/(TN+FN) = {TN}/{TN+FN} = {precision_down:.3f}')
        print(f'    → 모델이 "하락" 이라 한 {TN+FN} 시점 중 실제 {TN} 개가 하락')

    print('\n=== 실제 하락 시점 상세 (TN + FP) ===')
    down_cases = bt_df[bt_df['actual_dir'] == 'down'].copy()
    down_cases['caught'] = (down_cases['pred_dir'] == 'down').map({True: '✓ 잡음', False: '✗ 놓침'})
    print(down_cases[['cutoff', 'cutoff_price', 'actual_ret_pct', 'pred_ret_pct', 'caught']].to_string(index=False))

    print('\n=== 손실 절감 효과 (모델이 하락 잡았을 때) ===')
    caught = down_cases[down_cases['pred_dir'] == 'down']
    missed = down_cases[down_cases['pred_dir'] == 'up']
    if len(caught) > 0:
        print(f'  잡은 하락 ({len(caught)}건) 평균 실제 수익률: {caught["actual_ret_pct"].mean():+.2f}%')
    if len(missed) > 0:
        print(f'  놓친 하락 ({len(missed)}건) 평균 실제 수익률: {missed["actual_ret_pct"].mean():+.2f}%  ⚠️ 손실 미경보')

## 16. 하락 9 사례 — 실제 vs Production 모델 예측 시각화

**의도**: Specificity 0.000 결과를 **시각으로 직접 확인**. 9개 하락 cutoff 각각 panel 로 분리해서 모델이 어떻게 하락을 놓쳤는지 한눈에 비교.

**구성** (3×3 subplot, 각 panel 마다 이중축):
- **왼쪽 Y축 (가격)**: 
  - 회색 점선 = cutoff 가격 기준선
  - 검은 실선 = cutoff 직전 60일 actual
  - **빨간 실선** = cutoff 이후 30일 actual (실제 하락 path)
  - **파란 실선** = 모델 예측 yhat (대부분 상승)
  - 파란 영역 = 모델 lower/upper 신뢰구간
- **오른쪽 Y축 (누적 수익률 %)**: cutoff 대비 변화
  - 빨간 점선 = 실제 cumulative return
  - 파란 점선 = 모델 예측 cumulative return
  - 0% 기준선

**해석 가이드**:
- 빨간선이 0% 아래로 가는 동안 파란선이 위로 가면 → 모델이 정반대 방향 예측
- 가장 큰 하락 (2018-09 -7.68%, 2025-12 -2.0% 등)에서 파란선이 얼마나 위로 빗나갔는지가 핵심
- 모든 9 panel 에서 같은 패턴이면 → 단순 bull-bias가 아닌 **모델 구조적 한계**

In [ ]:
if not bt_records:
    print('backtest 결과 없음 — cell 25 먼저 실행')
else:
    down_records = [r for r in bt_records
                    if len(r['actual_prices']) >= BT_HORIZON
                    and r['actual_prices'][-1] < r['cutoff_price']]
    n_total = len(down_records)
    n_show = min(30, n_total)
    print(f'전체 하락 사례: {n_total}건 (표시: {n_show}건)')

    # 자동 grid (5xN 또는 6xN)
    cols = 5
    rows = (n_show + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 4, rows * 3.2))
    axes = axes.flatten() if rows * cols > 1 else [axes]

    # 큰 하락 순으로 정렬 (가장 큰 손실부터)
    down_records_sorted = sorted(down_records,
                                  key=lambda r: (r['actual_prices'][-1] / r['cutoff_price'] - 1))[:n_show]

    for i, r in enumerate(down_records_sorted):
        ax1 = axes[i]
        cp = r['cutoff_price']
        past = close_full.loc[close_full.index <= r['cutoff']].iloc[-60:]
        future_dates = r['actual_dates']
        future_prices = r['actual_prices']
        pred_dates = r['pred_dates']
        pred_yhat = r['pred_yhat']
        pred_lower = r['pred_lower']
        pred_upper = r['pred_upper']

        # 왼쪽 Y축 — 가격
        ax1.plot(past.index, past.values, color='#1f2937', lw=1.0)
        ax1.plot(future_dates, future_prices, color='#EF4444', lw=1.4)
        ax1.plot(pred_dates, pred_yhat, color='#3B82F6', lw=1.2)
        ax1.fill_between(pred_dates, pred_lower, pred_upper, color='#3B82F6', alpha=0.13)
        ax1.axvline(r['cutoff'], color='gray', ls='--', alpha=0.5)
        ax1.axhline(cp, color='gray', ls=':', alpha=0.35)
        ax1.tick_params(axis='y', labelsize=7)
        ax1.tick_params(axis='x', labelsize=6, rotation=45)
        ax1.grid(alpha=0.2)

        # 오른쪽 Y축 — 누적 수익률 %
        ax2 = ax1.twinx()
        actual_ret_pct = (np.array(future_prices) / cp - 1) * 100
        pred_ret_pct = (np.array(pred_yhat) / cp - 1) * 100
        ax2.plot(future_dates, actual_ret_pct, color='#EF4444', lw=0.7, alpha=0.5, ls=':')
        ax2.plot(pred_dates, pred_ret_pct, color='#3B82F6', lw=0.7, alpha=0.5, ls=':')
        ax2.axhline(0, color='gray', lw=0.3, alpha=0.5)
        ax2.tick_params(axis='y', labelsize=7)

        actual_ret_30d = (future_prices[-1] / cp - 1) * 100
        pred_ret_30d = (pred_yhat[-1] / cp - 1) * 100
        gap = pred_ret_30d - actual_ret_30d
        title = f'{r["cutoff"].strftime("%Y-%m")} | a:{actual_ret_30d:+.1f}% p:{pred_ret_30d:+.1f}% (gap {gap:+.1f}%p)'
        ax1.set_title(title, fontsize=8)

    for j in range(n_show, rows * cols):
        axes[j].axis('off')

    # 첫 panel 에만 범례
    axes[0].legend(['actual past', 'actual future', 'model pred', 'pred CI'], loc='upper left', fontsize=6)

    plt.suptitle(f'{TICKER_BT} 하락 {n_show} 사례 (큰 손실순) — 실제(빨강) vs Production 모델(파랑)',
                  y=1.0, fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.show()

    # 빗나간 폭 통계
    gaps = [(r['pred_yhat'][-1] / r['cutoff_price'] - 1) * 100 -
            (r['actual_prices'][-1] / r['cutoff_price'] - 1) * 100
            for r in down_records]
    print(f'\n=== 전체 {n_total} 하락 사례에서 모델 빗나간 폭 (pred%p - actual%p) ===')
    print(f'  mean : {np.mean(gaps):+.2f}%p  (모든 하락에서 모델이 평균 +{np.mean(gaps):.2f}%p 낙관 편향)')
    print(f'  max  : {np.max(gaps):+.2f}%p  / min: {np.min(gaps):+.2f}%p')
    print(f'  std  : {np.std(gaps):.2f}%p')
    print(f'  실제로 잡은 하락 (pred 도 음수): {sum(1 for r in down_records if r["pred_yhat"][-1] < r["cutoff_price"])}건 / {n_total}건')

## 17. Stage 1+2 적용 — 같은 30 cutoff 에 GJR-GARCH-Skew-t + FHS 1000 path

**의도**: production `_garch_forecast` (단일 σ, GARCH-t, 5y window) 가 30 cutoff 에서 mean prediction 으로 거의 모두 빗나갔다. 같은 cutoff 에 **plan stage 1+2 (GJR-GARCH(1,1,1)-Skew-t + 1y window + FHS Monte Carlo 1000 path)** 를 적용해서 분위수가 실제 하락을 얼마나 잡는지 비교.

**변경점 vs production**:
- 5-모델 앙상블 → XGBoost 단독 (mean path 만 빠르게 산출, GARCH 가 분산 담당이라 mean 자체는 단순화 OK)
- GARCH(1,1)-t (5y) → GJR-GARCH(1,1,1)-Skew-t (1y, 252일)
- 단일 σ forecast → FHS Monte Carlo 1000 path
- Hard clipping (`daily_clip`, `price_band`, 3-day MA) 모두 제거

**저장**: `stage12_records` — 각 cutoff 의 p05/p10/p50/p90/p95 price path + metrics(var_5pct, prob_down_5pct).

**비용**: 매 cutoff XGB 학습 + GJR-GARCH fit ≈ 3-8초. 120 cutoff = 6-15분 (production 5모델 앙상블의 3-5배 빠름).

In [ ]:
GARCH_WINDOW_BT = 252
N_SIMS_BT = 1000

stage12_records = []
print(f'Stage 1+2 backtest 시작 ({len(bt_records)} cutoff)...')
for i, r in enumerate(bt_records):
    cutoff = r['cutoff']
    cc = close_full.loc[close_full.index <= cutoff]
    if len(cc) < 300:
        continue
    try:
        # 1) XGBoost 단독 학습 (mean path)
        feat_t = build_features(cc)
        tgt_t = np.log(cc / cc.shift(1)).shift(-1)
        tgt_t.name = 'target'
        data_t = feat_t.join(tgt_t).dropna()
        if len(data_t) < 200:
            continue
        fcols = list(feat_t.columns)
        m = XGBRegressor(n_estimators=300, max_depth=4, learning_rate=0.05,
                          subsample=0.8, colsample_bytree=0.8, random_state=42)
        m.fit(data_t[fcols], data_t['target'], verbose=False)

        # 2) recursive mean path
        mu_path_bt = recursive_mean_forecast(m, cc, BT_HORIZON, fcols)

        # 3) GJR-GARCH-Skew-t (1년 윈도우)
        log_ret_pct = (np.log(cc / cc.shift(1)).dropna()) * 100
        log_ret_w = log_ret_pct.iloc[-GARCH_WINDOW_BT:]
        am_bt = arch_model(log_ret_w, vol='Garch', p=1, o=1, q=1, mean='Zero', dist='skewt')
        res_bt = am_bt.fit(disp='off', show_warning=False)

        # 4) FHS Monte Carlo
        paths_bt = fhs_monte_carlo(res_bt, mu_path_bt, N_SIMS_BT, BT_HORIZON)
        S0_bt = float(cc.iloc[-1])
        price_paths_bt = S0_bt * np.exp(np.cumsum(paths_bt, axis=1))

        # 5) 분위수 + metrics
        p05 = np.percentile(price_paths_bt, 5, axis=0)
        p10 = np.percentile(price_paths_bt, 10, axis=0)
        p50 = np.percentile(price_paths_bt, 50, axis=0)
        p90 = np.percentile(price_paths_bt, 90, axis=0)
        p95 = np.percentile(price_paths_bt, 95, axis=0)
        last_p = price_paths_bt[:, -1]
        ret = last_p / S0_bt - 1

        stage12_records.append({
            'cutoff': cutoff,
            'cutoff_price': S0_bt,
            'pred_dates': r['pred_dates'],
            'p05': p05, 'p10': p10, 'p50': p50, 'p90': p90, 'p95': p95,
            'metrics': {
                'var_5pct_30d': float(np.percentile(ret, 5)),
                'var_10pct_30d': float(np.percentile(ret, 10)),
                'expected_return_30d': float(np.median(ret)),
                'prob_down_5pct_30d': float((ret < -0.05).mean()),
                'prob_down_10pct_30d': float((ret < -0.10).mean()),
                'prob_up_30d': float((last_p > S0_bt).mean()),
            },
            'actual_dates': r['actual_dates'],
            'actual_prices': r['actual_prices'],
        })
        if (i + 1) % 20 == 0:
            print(f'  [{i+1}/{len(bt_records)}] {cutoff.date()} 완료')
    except Exception as e:
        print(f'  [{i+1}] {cutoff.date()} 실패: {e}')
        continue

print(f'\n총 {len(stage12_records)} 시점 stage 1+2 완료')

Stage 1+2 backtest 시작 (119 cutoff)...


## 18. Stage 1+2 시각화 — 새 lower CI 가 큰 하락을 잡는가?

**의도**: 같은 30개 큰 손실 cutoff 를 stage 1+2 분위수로 시각화. **production 단일 CI 와 비교**해서 새 lower CI(p05/p10) 가 actual 하락을 더 잘 잡는지 시각 확인.

**구성** (5×6 grid, 큰 손실 순):
- 검은 실선 — past 60일 actual
- **빨간 실선** — future 30일 actual (실제 하락)
- **빨간 영역 (p05~p10)** — 위험 zone (Stage 4 chart.js 강조 대상)
- 옅은 파랑 영역 (p10~p50) — 하한 구간
- 옅은 회색 영역 (p50~p95) — 상한 (덜 중요)
- **회색 얇은 선** — median (mean 자체는 강조 X)

**판정 기준**:
- 실제 빨간선이 **빨간 영역(p05~p10) 까지 떨어지면** → stage 1+2 가 위험을 사전 신호로 잡음 (good)
- 빨간선이 빨간 영역 더 아래로 뚫고 내려가면 → 여전히 fat-tail 부족 (bad)
- production 의 단일 CI 보다 lower bound 가 더 크게 내려가야 의미 있음

In [ ]:
if not stage12_records:
    print('stage12_records 없음 — cell 33 먼저 실행')
else:
    # 큰 손실순 (actual 30d return 기준)
    down_s12 = sorted(
        [r for r in stage12_records if len(r['actual_prices']) >= BT_HORIZON
         and r['actual_prices'][-1] < r['cutoff_price']],
        key=lambda r: r['actual_prices'][-1] / r['cutoff_price'] - 1
    )[:30]
    n_show = len(down_s12)
    print(f'표시할 하락 사례: {n_show}건')

    cols = 5
    rows = (n_show + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 4, rows * 3.2))
    axes = axes.flatten() if rows * cols > 1 else [axes]

    for i, r in enumerate(down_s12):
        ax = axes[i]
        cp = r['cutoff_price']
        past = close_full.loc[close_full.index <= r['cutoff']].iloc[-60:]
        fd, fp = r['actual_dates'], r['actual_prices']
        pd_ = r['pred_dates']

        # past + actual future
        ax.plot(past.index, past.values, color='#1f2937', lw=1.0)
        ax.plot(fd, fp, color='#EF4444', lw=1.6, zorder=5)
        # 분위수 영역 (4-layer)
        ax.fill_between(pd_, r['p50'], r['p95'], color='gray', alpha=0.10)
        ax.fill_between(pd_, r['p10'], r['p50'], color='#3B82F6', alpha=0.13)
        ax.fill_between(pd_, r['p05'], r['p10'], color='#EF4444', alpha=0.30)
        # median (옅은 회색)
        ax.plot(pd_, r['p50'], color='#9ca3af', lw=0.9, alpha=0.7)

        ax.axvline(r['cutoff'], color='gray', ls='--', alpha=0.5)
        ax.axhline(cp, color='gray', ls=':', alpha=0.35)
        ax.tick_params(axis='both', labelsize=7)
        plt.setp(ax.get_xticklabels(), rotation=45)
        ax.grid(alpha=0.2)

        a30 = (fp[-1] / cp - 1) * 100
        p05_30 = (r['p05'][-1] / cp - 1) * 100
        caught = '✓' if fp[-1] >= r['p05'][-1] else '✗'
        ax.set_title(f'{r["cutoff"].strftime("%Y-%m")} | actual {a30:+.1f}% / p05 {p05_30:+.1f}% {caught}',
                      fontsize=8)

    for j in range(n_show, rows * cols):
        axes[j].axis('off')

    plt.suptitle('Stage 1+2 — 빨간 영역(p05~p10) = 위험 zone, 실제 빨간선이 그 안/위 인지 확인',
                 y=1.0, fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.show()

    # 통계: stage 1+2 가 actual 을 p05 안에 넣은 비율
    inside_p05 = sum(1 for r in down_s12 if r['actual_prices'][-1] >= r['p05'][-1])
    inside_p10 = sum(1 for r in down_s12 if r['actual_prices'][-1] >= r['p10'][-1])
    print(f'\n=== Stage 1+2 calibration on {n_show} 하락 사례 ===')
    print(f'  actual >= p05 (95% 신뢰구간 안): {inside_p05}/{n_show} = {inside_p05/n_show:.1%}  (이상적: 95%)')
    print(f'  actual >= p10 (90% 신뢰구간 안): {inside_p10}/{n_show} = {inside_p10/n_show:.1%}  (이상적: 90%)')

## 19. Stage 3 메트릭 시계열 — prob_down_5pct, var_5pct

**의도**: stage 1+2 가 산출하는 **단일 점수 메트릭**을 시간 흐름에 따라 시각화. 사용자에게 mean prediction 보다 더 의사결정 가능한 정보 제공.

**비교 panel** (3개 axis):
1. **위 axis**: actual 30일 수익률 (실제 결과) — 검은 선
2. **중간 axis**: Stage 1+2 의 `expected_return_30d` (median path) — 파란 선
3. **아래 axis**: **`prob_down_5pct_30d`** (5% 이상 하락할 확률) — 빨간 막대

**해석 가이드**:
- 실제 큰 하락 시점 (2018-09, 2022 베어, 2025 베어) 에 빨간 막대가 spike 하면 — fan chart 가 사전 위험 신호 줌 (good)
- 막대가 항상 일정 (예: 항상 10%) 이면 — 메트릭이 시점 무관하게 평탄 (bad)
- median path 와 actual 의 상관성도 같이 봄 — 0이면 mean 무용지물 다시 확인

In [ ]:
if not stage12_records:
    print('stage12_records 없음 — cell 33 먼저 실행')
else:
    df_metrics = pd.DataFrame([{
        'cutoff': r['cutoff'],
        'actual_30d_pct': (r['actual_prices'][-1] / r['cutoff_price'] - 1) * 100
                            if len(r['actual_prices']) >= BT_HORIZON else None,
        'expected_return_30d_pct': r['metrics']['expected_return_30d'] * 100,
        'var_5pct_30d_pct': r['metrics']['var_5pct_30d'] * 100,
        'prob_down_5pct': r['metrics']['prob_down_5pct_30d'] * 100,
        'prob_down_10pct': r['metrics']['prob_down_10pct_30d'] * 100,
        'prob_up_30d': r['metrics']['prob_up_30d'] * 100,
    } for r in stage12_records]).dropna(subset=['actual_30d_pct'])

    fig, axes = plt.subplots(3, 1, figsize=(16, 10), sharex=True)

    # axis 1: actual 30d return
    actual_colors = ['#EF4444' if x < 0 else '#10B981' for x in df_metrics['actual_30d_pct']]
    axes[0].bar(df_metrics['cutoff'], df_metrics['actual_30d_pct'], width=20,
                 color=actual_colors, alpha=0.7)
    axes[0].axhline(0, color='gray', lw=0.5)
    axes[0].set_ylabel('Actual 30d return %', fontsize=10)
    axes[0].set_title('실제 30일 후 수익률 (red=하락, green=상승)', fontsize=11)
    axes[0].grid(alpha=0.3)

    # axis 2: expected return + VaR 5%
    axes[1].plot(df_metrics['cutoff'], df_metrics['expected_return_30d_pct'],
                  color='#3B82F6', lw=1.5, label='median (expected)', marker='o', markersize=3)
    axes[1].fill_between(df_metrics['cutoff'],
                           df_metrics['var_5pct_30d_pct'],
                           df_metrics['expected_return_30d_pct'],
                           color='#EF4444', alpha=0.15, label='median ~ VaR 5%')
    axes[1].plot(df_metrics['cutoff'], df_metrics['var_5pct_30d_pct'],
                  color='#EF4444', lw=1.0, ls='--', label='VaR 5%')
    axes[1].axhline(0, color='gray', lw=0.5)
    axes[1].set_ylabel('Predicted 30d return %', fontsize=10)
    axes[1].set_title('Stage 1+2 예측 — median + VaR 5% (5% 확률 손실)', fontsize=11)
    axes[1].legend(fontsize=9)
    axes[1].grid(alpha=0.3)

    # axis 3: prob_down_5pct + prob_down_10pct
    axes[2].bar(df_metrics['cutoff'], df_metrics['prob_down_5pct'], width=20,
                 color='#EF4444', alpha=0.5, label='prob_down_5pct (5% 이상 하락 확률)')
    axes[2].plot(df_metrics['cutoff'], df_metrics['prob_down_10pct'],
                  color='#7c1d1d', lw=1.0, marker='s', markersize=3,
                  label='prob_down_10pct (10% 이상 하락 확률)')
    axes[2].axhline(50, color='gray', lw=0.3, ls=':')
    axes[2].set_ylabel('Probability %', fontsize=10)
    axes[2].set_title('Stage 3 핵심 메트릭 — 하락 확률 (사용자에게 직접 보여줄 정보)', fontsize=11)
    axes[2].legend(fontsize=9)
    axes[2].grid(alpha=0.3)

    plt.suptitle(f'{TICKER_BT} — Stage 1+2 메트릭 시계열 (10년 walk-forward, {len(df_metrics)} cutoff)',
                  y=1.0, fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.show()

    # 상관관계 — prob_down_5pct 가 실제 큰 하락을 사전 예고하는가?
    bad_cutoffs = df_metrics[df_metrics['actual_30d_pct'] < -5]
    good_cutoffs = df_metrics[df_metrics['actual_30d_pct'] > 0]
    print('\n=== prob_down_5pct 가 사전 신호인가? ===')
    print(f'  실제 -5% 이상 하락한 시점 ({len(bad_cutoffs)}건) 의 평균 prob_down_5pct: {bad_cutoffs["prob_down_5pct"].mean():.1f}%')
    print(f'  실제 +상승한 시점 ({len(good_cutoffs)}건) 의 평균 prob_down_5pct: {good_cutoffs["prob_down_5pct"].mean():.1f}%')
    print(f'  → 두 그룹의 prob_down 차이가 클수록 메트릭이 의미 있음')

    corr = df_metrics[['actual_30d_pct', 'expected_return_30d_pct', 'var_5pct_30d_pct',
                         'prob_down_5pct', 'prob_down_10pct']].corr()['actual_30d_pct'].drop('actual_30d_pct')
    print('\n=== actual_30d_pct 와 메트릭의 상관관계 ===')
    print(corr.round(3))
    print('  (양수 클수록 정상 신호 — expected/median 은 positive 가능, prob_down 은 negative 가 정상)')

## 20. Stage 4 chart.js 색상 스킴 — 위험 영역 강조

**의도**: 본 코드 `static/js/chart.js` SVG 렌더링이 사용자에게 보여줄 **최종 fan chart 모습**을 matplotlib 으로 prototype.

**핵심 변경 — 색상 스킴 재설계**:
- **mean line** → 옅은 회색 얇은 선 (현재 파란 굵은선에서 강조 대거 약화)
- **p05~p10 영역** → **빨간 fill (위험 zone)** — 사용자가 한눈에 "여기 떨어지면 위험"
- **p10~p50 영역** → 옅은 파랑 (하한)
- **p50~p95 영역** → 옅은 회색/청록 (상한, 덜 중요)
- **median path** → 옅은 회색 점선 (mean 안 강조)
- 우상단 **VaR 5% / prob_down_5pct 메트릭 텍스트** 표시

**색채 의도**:
- 빨강 = '주의 (potential loss)' — 시각적으로 즉각 경계
- 파랑/회색 = '중립적 가능 범위'
- mean 강조 X — 어차피 점예측 무용지물

**비교 layout** (12개만 — 큰 하락 위주):
- 왼쪽: production 단일 CI (참고)
- 오른쪽: stage 1+2 fan + 위험 영역
- 같은 cutoff 옆에 나란히

In [ ]:
if not stage12_records or not bt_records:
    print('데이터 없음')
else:
    # cutoff 키로 매칭
    s12_by_cutoff = {r['cutoff']: r for r in stage12_records}
    pairs = []
    for r in sorted(bt_records,
                     key=lambda r: r['actual_prices'][-1] / r['cutoff_price'] - 1
                     if len(r['actual_prices']) >= BT_HORIZON else 0):
        if (len(r['actual_prices']) >= BT_HORIZON
            and r['actual_prices'][-1] < r['cutoff_price']
            and r['cutoff'] in s12_by_cutoff):
            pairs.append((r, s12_by_cutoff[r['cutoff']]))
        if len(pairs) >= 12:
            break

    n = len(pairs)
    print(f'표시: {n}쌍 (production vs stage 1+2)')

    fig, axes = plt.subplots(n, 2, figsize=(14, n * 2.5))
    if n == 1:
        axes = axes.reshape(1, 2)

    for i, (r_old, r_new) in enumerate(pairs):
        cp = r_old['cutoff_price']
        past = close_full.loc[close_full.index <= r_old['cutoff']].iloc[-60:]
        fd, fp = r_old['actual_dates'], r_old['actual_prices']

        # 왼쪽: production (변경 전)
        ax = axes[i, 0]
        ax.plot(past.index, past.values, color='#1f2937', lw=1.0)
        ax.plot(fd, fp, color='#EF4444', lw=1.5, zorder=5)
        ax.fill_between(r_old['pred_dates'], r_old['pred_lower'], r_old['pred_upper'],
                          color='#3B82F6', alpha=0.15)
        ax.plot(r_old['pred_dates'], r_old['pred_yhat'], color='#3B82F6', lw=1.5)
        ax.axvline(r_old['cutoff'], color='gray', ls='--', alpha=0.5)
        ax.set_title(f'{r_old["cutoff"].strftime("%Y-%m")} | Production (현재)',
                      fontsize=9)
        ax.tick_params(axis='both', labelsize=7)
        plt.setp(ax.get_xticklabels(), rotation=45)
        ax.grid(alpha=0.2)

        # 오른쪽: stage 1+2 (제안)
        ax = axes[i, 1]
        pd_ = r_new['pred_dates']
        ax.plot(past.index, past.values, color='#1f2937', lw=1.0)
        ax.plot(fd, fp, color='#EF4444', lw=1.5, zorder=5)
        # 색상 스킴: 위험 영역 빨강, 중립 회색, 상한 옅은 회색
        ax.fill_between(pd_, r_new['p50'], r_new['p95'], color='#9ca3af', alpha=0.12)
        ax.fill_between(pd_, r_new['p10'], r_new['p50'], color='#60a5fa', alpha=0.15)
        ax.fill_between(pd_, r_new['p05'], r_new['p10'], color='#EF4444', alpha=0.32, label='위험 (p05~p10)')
        ax.plot(pd_, r_new['p50'], color='#9ca3af', lw=0.8, ls='--', alpha=0.7)
        ax.axvline(r_new['cutoff'], color='gray', ls='--', alpha=0.5)

        # 우상단 메트릭
        m = r_new['metrics']
        info = (f"VaR5%: {m['var_5pct_30d']*100:+.1f}%\n"
                 f"P[down 5%]: {m['prob_down_5pct_30d']*100:.0f}%\n"
                 f"P[down 10%]: {m['prob_down_10pct_30d']*100:.0f}%")
        ax.text(0.97, 0.97, info, transform=ax.transAxes, fontsize=7,
                 ha='right', va='top', bbox=dict(facecolor='white', alpha=0.85, edgecolor='gray'))
        ax.set_title(f'{r_new["cutoff"].strftime("%Y-%m")} | Stage 1+2 (제안)', fontsize=9)
        ax.tick_params(axis='both', labelsize=7)
        plt.setp(ax.get_xticklabels(), rotation=45)
        ax.grid(alpha=0.2)
        if i == 0:
            ax.legend(fontsize=7, loc='lower left')

    plt.suptitle(f'{TICKER_BT} — Production (왼쪽) vs Stage 1+2 (오른쪽). 빨간 영역 = 위험 zone',
                  y=1.0, fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.show()

## 21. Stage 5 — Conformal Prediction Calibration

**문제**: Stage 1+2 만으로는 lower CI 가 큰 하락을 못 잡음 (p05 coverage 73.3%, 이상 95%). Conformal prediction 으로 명목 coverage 강제 보정.

**Split Conformal 핵심 아이디어**:
1. Walk-forward 직전 K개 cutoff (calibration set)에서 잔차 산출:
   `residual = actual_30d_return - predicted_p05_return`
2. 잔차 분포의 5% 분위수 = `adj` (음수면 under-coverage = pred_p05 가 너무 낙관)
3. 새 cutoff 의 p05 를 `adj` 만큼 추가 하향 → **명목 95% coverage 통계적으로 보장**

**왜 효과 있나?** — 분포 형태(GARCH/Skew-t)와 무관하게 **historical miss 패턴**을 직접 학습. 모델이 일관되게 X% 빗나갔다면 다음번엔 그만큼 미리 보정.

**Trade-off**:
- 장점: distribution-free, 통계적 보장 (대수의 법칙)
- 단점: regime change 직후엔 calibration 지연 (직전 K cutoff 의 잔차가 옛 regime 기준)
- 처음 K cutoff 는 calibration 데이터 부족으로 보정 불가 (warm-up)

In [ ]:
if not stage12_records:
    print('stage12_records 없음')
else:
    CALIB_WINDOW = 30   # 직전 30 cutoff 로 calibration

    sorted_recs = sorted([r for r in stage12_records if len(r['actual_prices']) >= BT_HORIZON],
                          key=lambda r: r['cutoff'])

    for i, r in enumerate(sorted_recs):
        cp = r['cutoff_price']
        if i < CALIB_WINDOW:
            r['p05_cf'] = r['p05'].copy()
            r['p10_cf'] = r['p10'].copy()
            r['adj_p05'] = 0.0
            r['adj_p10'] = 0.0
            r['var_5pct_cf'] = r['metrics']['var_5pct_30d']
            r['prob_down_5pct_cf'] = r['metrics']['prob_down_5pct_30d']
            continue

        cal = sorted_recs[i-CALIB_WINDOW:i]
        res_p05 = [cr['actual_prices'][-1] / cr['cutoff_price'] - cr['p05'][-1] / cr['cutoff_price']
                   for cr in cal]
        res_p10 = [cr['actual_prices'][-1] / cr['cutoff_price'] - cr['p10'][-1] / cr['cutoff_price']
                   for cr in cal]

        # 5% / 10% quantile of residuals — 음수만 보정 (under-coverage 만)
        adj_p05 = min(0, np.quantile(res_p05, 0.05))
        adj_p10 = min(0, np.quantile(res_p10, 0.10))
        r['adj_p05'] = adj_p05
        r['adj_p10'] = adj_p10

        # path 전체에 동일 비율 보정
        r['p05_cf'] = r['p05'] * (1 + adj_p05)
        r['p10_cf'] = r['p10'] * (1 + adj_p10)

        # 보정된 VaR / prob_down 도 재산출
        r['var_5pct_cf'] = r['metrics']['var_5pct_30d'] + adj_p05
        # prob_down 보정: 음수 adj 로 분포 좌측 이동 → prob_down 증가
        # 단순화: 기존 prob_down 에 adj 비율의 영향 더함
        from scipy.stats import norm
        # 기존 분포 std 추정 (대략): (median - p05) / 1.645 ≈ sigma
        med_ret = r['metrics']['expected_return_30d']
        p05_ret = r['p05'][-1] / cp - 1
        sigma = max(1e-6, (med_ret - p05_ret) / 1.645)
        # 보정 후 P(ret < -0.05)
        new_p05_ret = p05_ret + adj_p05
        new_med_ret = med_ret  # median 은 안 보정
        new_sigma = max(1e-6, (new_med_ret - new_p05_ret) / 1.645)
        r['prob_down_5pct_cf'] = float(norm.cdf((-0.05 - new_med_ret) / new_sigma))

    # === Calibration 비교 ===
    eval_recs = sorted_recs[CALIB_WINDOW:]
    n_eval = len(eval_recs)
    in_p05_old = sum(1 for r in eval_recs if r['actual_prices'][-1] >= r['p05'][-1])
    in_p05_cf = sum(1 for r in eval_recs if r['actual_prices'][-1] >= r['p05_cf'][-1])
    in_p10_old = sum(1 for r in eval_recs if r['actual_prices'][-1] >= r['p10'][-1])
    in_p10_cf = sum(1 for r in eval_recs if r['actual_prices'][-1] >= r['p10_cf'][-1])

    print(f'평가 시점: {n_eval} (warm-up {CALIB_WINDOW} 이후)')
    print(f'\n=== p05 (95% 신뢰구간) coverage ===')
    print(f'  Stage 1+2:           {in_p05_old}/{n_eval} = {in_p05_old/n_eval:.1%}')
    print(f'  Stage 5 conformal:   {in_p05_cf}/{n_eval} = {in_p05_cf/n_eval:.1%}  (이상: 95%)')
    print(f'\n=== p10 (90% 신뢰구간) coverage ===')
    print(f'  Stage 1+2:           {in_p10_old}/{n_eval} = {in_p10_old/n_eval:.1%}')
    print(f'  Stage 5 conformal:   {in_p10_cf}/{n_eval} = {in_p10_cf/n_eval:.1%}  (이상: 90%)')

    # === prob_down_5pct 신호 — 보정 후 다시 평가 ===
    df_cf = pd.DataFrame([{
        'cutoff': r['cutoff'],
        'actual_30d_pct': (r['actual_prices'][-1] / r['cutoff_price'] - 1) * 100,
        'prob_down_5pct_old': r['metrics']['prob_down_5pct_30d'] * 100,
        'prob_down_5pct_cf': r['prob_down_5pct_cf'] * 100,
    } for r in eval_recs])

    bad = df_cf[df_cf['actual_30d_pct'] < -5]
    good = df_cf[df_cf['actual_30d_pct'] > 0]
    print(f'\n=== prob_down_5pct 신호 정상화 ===')
    print(f'  실제 큰 하락 시점 ({len(bad)}건):')
    print(f'    Stage 1+2:    avg prob_down = {bad["prob_down_5pct_old"].mean():.1f}%')
    print(f'    Stage 5:      avg prob_down = {bad["prob_down_5pct_cf"].mean():.1f}%')
    print(f'  실제 상승 시점 ({len(good)}건):')
    print(f'    Stage 1+2:    avg prob_down = {good["prob_down_5pct_old"].mean():.1f}%')
    print(f'    Stage 5:      avg prob_down = {good["prob_down_5pct_cf"].mean():.1f}%')
    print(f'  → 두 그룹 차이 (큰값 - 작은값): old={bad["prob_down_5pct_old"].mean() - good["prob_down_5pct_old"].mean():+.1f}%p, cf={bad["prob_down_5pct_cf"].mean() - good["prob_down_5pct_cf"].mean():+.1f}%p')

    corr_old = df_cf[['actual_30d_pct', 'prob_down_5pct_old']].corr().iloc[0,1]
    corr_cf = df_cf[['actual_30d_pct', 'prob_down_5pct_cf']].corr().iloc[0,1]
    print(f'\n  상관관계 (actual vs prob_down):')
    print(f'    Stage 1+2:  {corr_old:+.3f}  (음수 = 정상 신호)')
    print(f'    Stage 5:    {corr_cf:+.3f}')

    # === 30 panel 시각화 — 큰 손실순 30개 ===
    down_eval = sorted([r for r in eval_recs if r['actual_prices'][-1] < r['cutoff_price']],
                        key=lambda r: r['actual_prices'][-1] / r['cutoff_price'] - 1)[:30]
    n_show = len(down_eval)
    cols = 5
    rows = (n_show + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(cols * 4, rows * 3.2))
    axes = axes.flatten() if rows * cols > 1 else [axes]

    for i, r in enumerate(down_eval):
        ax = axes[i]
        cp = r['cutoff_price']
        past = close_full.loc[close_full.index <= r['cutoff']].iloc[-60:]
        fd, fp = r['actual_dates'], r['actual_prices']
        pd_ = r['pred_dates']

        ax.plot(past.index, past.values, color='#1f2937', lw=1.0)
        ax.plot(fd, fp, color='#EF4444', lw=1.6, zorder=6)
        # Stage 1+2 p05 — 옅은 빨강
        ax.fill_between(pd_, r['p05'], r['p10'], color='#FCA5A5', alpha=0.30, label='S1+2 p05~p10')
        # Stage 5 conformal p05 — 진한 빨강
        ax.fill_between(pd_, r['p05_cf'], r['p10_cf'], color='#DC2626', alpha=0.30, label='S5 conformal')
        ax.plot(pd_, r['p50'], color='#9ca3af', lw=0.8, ls='--', alpha=0.7)
        ax.axvline(r['cutoff'], color='gray', ls='--', alpha=0.5)
        ax.tick_params(axis='both', labelsize=7)
        plt.setp(ax.get_xticklabels(), rotation=45)
        ax.grid(alpha=0.2)

        a30 = (fp[-1] / cp - 1) * 100
        p05_old_pct = (r['p05'][-1] / cp - 1) * 100
        p05_cf_pct = (r['p05_cf'][-1] / cp - 1) * 100
        c_old = '✓' if fp[-1] >= r['p05'][-1] else '✗'
        c_cf = '✓' if fp[-1] >= r['p05_cf'][-1] else '✗'
        ax.set_title(f'{r["cutoff"].strftime("%Y-%m")} a:{a30:+.1f}% | s12 p05 {p05_old_pct:+.1f}% {c_old} | s5 p05 {p05_cf_pct:+.1f}% {c_cf}',
                      fontsize=7)
        if i == 0:
            ax.legend(fontsize=6, loc='lower left')

    for j in range(n_show, rows * cols):
        axes[j].axis('off')

    plt.suptitle(f'Stage 5 Conformal — 옅은 빨강 = S1+2 p05, 진한 빨강 = conformal-보정 p05',
                  y=1.0, fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.show()

## 22. 최종 Production 적용 후 모습 — 사용자가 실제로 보는 차트

**확정된 production 사양** (이전 단계 결과 종합):
- **Stage 1+2 본 코드 적용 OK** — calibration 93% 통과 (89 시점 평가)
- **Stage 4 chart.js 가 가장 큰 사용자 가치** — fan chart 시각 개선
- **Stage 5 conformal 적용 X** — 효과 거의 0, 복잡도만 증가
- **prob_down_5pct 메트릭 강조 X** — 신호 약함. 대신 **VaR 5%/10%** 강조 (calibration OK)
- **mean line 강조 X** — bull-bias 무용지물

**구성**:
- **왼쪽 chart — 오늘 시점 production**: 사용자가 사이트에서 실제 보는 모습
- **오른쪽 chart — historical 검증** (2022-03 베어마켓 시작 시점): 모델이 그 당시 어떤 fan 그렸을지

**4-layer fan**:
- p50~p95: 옅은 회색 (상한, 덜 중요)
- p10~p50: 옅은 파랑 (중립)
- **p05~p10: 빨강 (⚠️ 위험 zone)** — 최대 강조
- median: 옅은 회색 점선 (강조 X)
- 30 sample paths: 매우 옅은 회색

**우상단 메트릭 박스**: VaR 5% / VaR 10% / 기대 수익률
**하단 disclaimer**: black swan 예측 불가 + 투자 권유 아님

In [ ]:
def production_chart(ax, ticker, close_hist, paths, S0, last_date, future_dates, subtitle=''):
    """Production fan chart — 최종 색상 스킴 + 메트릭 박스 + disclaimer."""
    past = close_hist.iloc[-90:]
    ax.plot(past.index, past.values, color='#1f2937', lw=1.4, label='Actual', zorder=10)

    p05 = np.percentile(paths, 5, axis=0)
    p10 = np.percentile(paths, 10, axis=0)
    p50 = np.percentile(paths, 50, axis=0)
    p90 = np.percentile(paths, 90, axis=0)
    p95 = np.percentile(paths, 95, axis=0)

    # 4-layer band (위험 zone 강조)
    ax.fill_between(future_dates, p50, p95, color='#9ca3af', alpha=0.10)
    ax.fill_between(future_dates, p90, p95, color='#9ca3af', alpha=0.06)
    ax.fill_between(future_dates, p10, p50, color='#60a5fa', alpha=0.13)
    ax.fill_between(future_dates, p05, p10, color='#EF4444', alpha=0.35,
                     label='⚠️ 위험 zone (p05~p10)')

    # 30 sample paths (옅은 회색 그림자)
    sample_idx = np.random.choice(paths.shape[0], 30, replace=False)
    for i in sample_idx:
        ax.plot(future_dates, paths[i], color='gray', alpha=0.05, lw=0.4)

    # Median (강조 X — 옅은 점선)
    ax.plot(future_dates, p50, color='#9ca3af', lw=1.1, ls='--', alpha=0.75, label='median (참고용)')

    ax.axvline(last_date, color='red', ls='--', alpha=0.5, lw=1)

    # 우상단 메트릭 박스
    last_p = paths[:, -1]
    ret = last_p / S0 - 1
    var_5 = np.percentile(ret, 5) * 100
    var_10 = np.percentile(ret, 10) * 100
    exp_ret = np.median(ret) * 100
    p95_ret = np.percentile(ret, 95) * 100

    info = ('30일 후 예상\n'
            '─────────────\n'
            f'기대 수익률    {exp_ret:+5.1f}%\n'
            f'5% 확률 손실   {var_5:+5.1f}%\n'
            f'10% 확률 손실  {var_10:+5.1f}%\n'
            f'5% 확률 상승   {p95_ret:+5.1f}%')
    ax.text(0.985, 0.985, info, transform=ax.transAxes, fontsize=9,
             ha='right', va='top', family='monospace',
             bbox=dict(facecolor='white', alpha=0.94, edgecolor='#9ca3af',
                       boxstyle='round,pad=0.6', linewidth=1))

    title = f'{ticker} — 30일 가격 분포 예측'
    if subtitle:
        title += f'  ({subtitle})'
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.set_ylabel('Price ($)')
    ax.legend(loc='upper left', fontsize=9, framealpha=0.92)
    ax.grid(alpha=0.25)
    plt.setp(ax.get_xticklabels(), rotation=30, fontsize=8)

# === 1. 오늘 시점 — production preview ===
# cell 16 의 paths_c (GJR-Skew-t 1y window FHS) 그대로 사용
future_dates_now = pd.bdate_range(close.index[-1] + pd.Timedelta(days=1), periods=HORIZON)

# === 2. Historical fail case — 2022-03 ===
# stage12_records 에서 2022-03 cutoff 매칭
if stage12_records:
    target = pd.Timestamp('2022-03-31')
    match = min(stage12_records, key=lambda r: abs((r['cutoff'] - target).days))
    cp_h = match['cutoff_price']
    # paths 재구성 — quantile 만 저장돼있어서 backfill 어려움. 대신 분위수로 fan 직접 그림
    has_historical = True
else:
    has_historical = False

if has_historical:
    fig, axes = plt.subplots(1, 2, figsize=(20, 7))
    production_chart(axes[0], TICKER, close, price_c, S0, close.index[-1],
                      future_dates_now, subtitle='오늘 production')

    # 오른쪽 — historical 시점 (2022-03 같은 베어 시작)
    ax = axes[1]
    cp = match['cutoff_price']
    past_h = close_full.loc[close_full.index <= match['cutoff']].iloc[-90:]
    ax.plot(past_h.index, past_h.values, color='#1f2937', lw=1.4, label='Actual (past)', zorder=10)

    # 실제 30일 후 (검증용 빨강 굵은선)
    fd, fp = match['actual_dates'], match['actual_prices']
    ax.plot(fd, fp, color='#dc2626', lw=2.0, label='Actual (실제 30일)', zorder=11)

    # Stage 1+2 분위수 fan
    pd_ = match['pred_dates']
    ax.fill_between(pd_, match['p50'], match['p95'], color='#9ca3af', alpha=0.10)
    ax.fill_between(pd_, match['p10'], match['p50'], color='#60a5fa', alpha=0.13)
    ax.fill_between(pd_, match['p05'], match['p10'], color='#EF4444', alpha=0.35,
                     label='⚠️ 위험 zone')
    ax.plot(pd_, match['p50'], color='#9ca3af', lw=1.1, ls='--', alpha=0.75, label='median')
    ax.axvline(match['cutoff'], color='red', ls='--', alpha=0.5)

    # 메트릭 박스
    m = match['metrics']
    a30 = (fp[-1] / cp - 1) * 100
    info_h = ('당시 30일 후 예상\n'
               '─────────────\n'
               f"기대 수익률    {m['expected_return_30d']*100:+5.1f}%\n"
               f"5% 확률 손실   {m['var_5pct_30d']*100:+5.1f}%\n"
               f"10% 확률 손실  {m['var_10pct_30d']*100:+5.1f}%\n"
               '─────────────\n'
               f'실제 30일 후:  {a30:+5.1f}%')
    ax.text(0.985, 0.985, info_h, transform=ax.transAxes, fontsize=9,
             ha='right', va='top', family='monospace',
             bbox=dict(facecolor='white', alpha=0.94, edgecolor='#9ca3af',
                       boxstyle='round,pad=0.6', linewidth=1))

    ax.set_title(f'{TICKER_BT} {match["cutoff"].strftime("%Y-%m")} — historical 검증 (베어 시작)',
                  fontsize=12, fontweight='bold')
    ax.set_ylabel('Price ($)')
    ax.legend(loc='upper left', fontsize=9, framealpha=0.92)
    ax.grid(alpha=0.25)
    plt.setp(ax.get_xticklabels(), rotation=30, fontsize=8)

    # Disclaimer
    fig.text(0.5, 0.01,
             '⚠️ 정상 시장 분포 추정. Black swan(코로나, 베어마켓 시작 등) 사전 예측 불가. 투자 권유 아님.',
             ha='center', fontsize=9.5, style='italic', color='#6b7280')
    plt.tight_layout(rect=[0, 0.03, 1, 1])
    plt.show()

    # === 텍스트 요약 ===
    print(f'\n=== Production 적용 시 사용자 화면 — 핵심 정보 ===')
    last_p = price_c[:, -1]
    ret_now = last_p / S0 - 1
    print(f'\n[오늘 {TICKER} 30일 fan]')
    print(f'  기대 수익률 (median):   {np.median(ret_now)*100:+.2f}%')
    print(f'  5% 확률 최대 손실:       {np.percentile(ret_now, 5)*100:+.2f}%')
    print(f'  10% 확률 최대 손실:      {np.percentile(ret_now, 10)*100:+.2f}%')
    print(f'  5% 확률 최대 상승:       {np.percentile(ret_now, 95)*100:+.2f}%')

    print(f'\n[2022-03 베어 시작 — 그때 사용자가 봤을 화면]')
    print(f"  기대 수익률 (median):   {m['expected_return_30d']*100:+.2f}%  (실제 +{a30:.2f}%? 사실은 {a30:+.2f}%)")
    print(f"  5% 확률 최대 손실:       {m['var_5pct_30d']*100:+.2f}%")
    print(f'  → 위험 zone 안에 actual {a30:+.1f}% 포함 여부: {"포함" if fp[-1] >= match["p05"][-1] else "이탈 (black swan)"}')
else:
    fig, ax = plt.subplots(figsize=(13, 7))
    production_chart(ax, TICKER, close, price_c, S0, close.index[-1],
                      future_dates_now, subtitle='오늘 production')
    fig.text(0.5, 0.01,
             '⚠️ 정상 시장 분포 추정. Black swan 사전 예측 불가. 투자 권유 아님.',
             ha='center', fontsize=9.5, style='italic', color='#6b7280')
    plt.tight_layout(rect=[0, 0.03, 1, 1])
    plt.show()